# Document vs Photograph Classification

Real document-like images and photographs.

## Step 1: Import libraries

This cell imports image, numerical, visualization and machine-learning tools.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


## Step 2: Locate the included real-image dataset

Each folder name is treated as one class.

In [ ]:
DATASET_DIR=Path("../datasets/04_document_vs_photo")
print([p.name for p in DATASET_DIR.iterdir() if p.is_dir()])

## Step 3: Load and resize images

Traditional ML needs consistent image dimensions.

In [ ]:
def load_images(path, size=(128,128)):
    images, labels = [], []
    for class_dir in sorted(path.iterdir()):
        if class_dir.is_dir():
            for fp in sorted(class_dir.glob("*.png")):
                images.append(np.array(Image.open(fp).convert("RGB").resize(size)))
                labels.append(class_dir.name)
    return np.array(images), np.array(labels)

images,labels=load_images(DATASET_DIR)
print(images.shape,labels.shape)

## Step 4: Display real image samples

This verifies the dataset.

In [ ]:
classes=sorted(set(labels))
plt.figure(figsize=(12,3))
for i,name in enumerate(classes,1):
    idx=np.where(labels==name)[0][0]
    plt.subplot(1,len(classes),i); plt.imshow(images[idx]); plt.title(name); plt.axis("off")
plt.tight_layout(); plt.show()


## Step 5: Extract features

HOG captures document stroke and layout patterns.

In [ ]:
from skimage.feature import hog
def hog_feat(im):
    gray=np.mean(im,axis=2)/255.0
    return hog(gray,orientations=9,pixels_per_cell=(16,16),cells_per_block=(2,2),block_norm="L2-Hys")
features=np.array([hog_feat(im) for im in images])
print(features.shape)


## Step 6: Split train and test sets

The test set remains unseen during training.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(features,labels,test_size=.25,random_state=42,stratify=labels)
print("Train:",len(X_train),"Test:",len(X_test))


## Step 7: Train the model

SVM separates the categories.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
model=make_pipeline(StandardScaler(),SVC(C=5))
model.fit(X_train,y_train)


## Step 8: Evaluate

We use accuracy, classification report and confusion matrix.

In [ ]:
pred=model.predict(X_test)
print("Accuracy:",round(accuracy_score(y_test,pred),4))
print(classification_report(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred,xticks_rotation=45)
plt.tight_layout(); plt.show()


## Conclusion

Workflow: real images → preprocessing → feature extraction → traditional ML → evaluation.